# Experimento A — Descripción del Dataset
**IIC3912 Tópicos Avanzados de Gráfica Computacional — Tarea 2**

# Instalación de dependencias y conexión con drive

In [ ]:
!pip install pycolmap plotly pandas -q

import os, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pycolmap
from pathlib import Path
from PIL import Image as PILImage
from IPython.display import display

print('pycolmap:', pycolmap.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.1/27.1 MB 38.6 MB/s eta 0:00:00
pycolmap: 4.0.4


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SCENES    = ['garden', 'bicycle', 'bonsai', 'counter']
IMG_SCALE = 4

DRIVE_BASE = Path('/content/drive/MyDrive/tarea2_mipnerf')
BASE       = Path('/content/colmap_work')
DRIVE_BASE.mkdir(parents=True, exist_ok=True)

def scene_paths(scene):
    return {
        'drive'    : DRIVE_BASE / scene,
        'images'   : BASE / scene / 'images',
        'sparse'   : BASE / scene / 'sparse',
        'db'       : BASE / scene / 'colmap.db',
        'official' : DRIVE_BASE / scene / 'sparse' / '0',
    }

for scene in SCENES:
    p = scene_paths(scene)
    for d in [p['images'], p['sparse'] / 'student', p['sparse'] / 'official']:
        d.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


In [ ]:
DATASET_URL = 'https://storage.googleapis.com/gresearch/refraw360/360_v2.zip'
ZIP_PATH    = DRIVE_BASE / '360_v2.zip'

def download_and_extract(scene):
    dst = DRIVE_BASE / scene
    if dst.exists():
        print(f'[{scene}] ya en Drive ✓')
        return
    if not ZIP_PATH.exists():
        print('Descargando dataset (~1.8 GB)...')
        ret = os.system(f'wget -q --show-progress -O "{ZIP_PATH}" "{DATASET_URL}"')
        if ret != 0:
            raise RuntimeError('Descarga fallida.')
    print(f'[{scene}] extrayendo...')
    os.system(f'unzip -q "{ZIP_PATH}" "{scene}/*" -d "{DRIVE_BASE}"')
    print(f'[{scene}] listo en {dst}')

def prepare_images(scene):
    paths = scene_paths(scene)
    dst   = paths['images']
    existing = list(dst.glob('*.jpg')) + list(dst.glob('*.JPG')) + list(dst.glob('*.png'))
    if existing:
        print(f'[{scene}] {len(existing)} imágenes ya en local ✓')
        return
    scale_folder = f'images_{IMG_SCALE}' if IMG_SCALE > 1 else 'images'
    src = paths['drive'] / scale_folder
    if not src.exists():
        src = paths['drive'] / 'images'
        print(f'[{scene}] usando images/ (sin downscale)')
    imgs = sorted(list(src.glob('*.jpg')) + list(src.glob('*.JPG')) + list(src.glob('*.png')))
    for img_path in imgs:
        shutil.copy(str(img_path), str(dst / img_path.name))
    with PILImage.open(list(dst.glob('*'))[0]) as im:
        print(f'[{scene}] {im.size} | {len(imgs)} imgs ✓')

for scene in SCENES:
    download_and_extract(scene)
    prepare_images(scene)

# (a) Descripción cualitativa de cada escena

Las cuatro escenas provienen de entornos reales y representan distintos desafíos para
Structure-from-Motion (SfM).

### Descripción y dificultades esperadas para SfM

| Escena | Tipo | Objeto principal | Características visuales | Dificultades para SfM |
|--------|------|-----------------|-------------------------|-----------------------|
| **garden** | Outdoor | Jardín con una mesa y un florero encima | plantas, piedras, esculturas, maceteros, pasto árboles, casas. Iluminación natural | Vegetación repetitiva puede confundir el matching, cambios de exposición entre tomas |
| **bicycle** | Outdoor | Bicicleta sobre pasto apoyado en una banca | Bicicleta blanca, banca de color negro, plantas, pasto, árboles, fondo poco definido | pocas features en zonas de pasto y cielo, es decir, casi no hay texturas |
| **bonsai** | Indoor | Árbol bonsai en habitación encima de una mesa | Ramas con texturas, flores rosadas, fondo con diversos elementos (bicicleta, piano, etc), superficies planas en pared y mesa con un mantel | Superficies planas con pocas features en paredes, hay mucho ruido en el fondo de la imagen donde se encuentran los objetos|
| **counter** | Indoor | Mesón de cocina con diversos alimentos y artículos de cocina | Artículos de cocina como bowl, ollas, entre otros. Alimentos como cebollas y naranjas. Mesón de cocina negro con brillantes | Hay mucho ruido debido a la cantidad de objetos en el mesón |


# (b) Estadísticas básicas de las imágenes

In [ ]:
def get_scene_imgs(scene):
    drive_dir = DRIVE_BASE / scene / f'images_{IMG_SCALE}'
    if not drive_dir.exists():
        drive_dir = DRIVE_BASE / scene / 'images'
    return sorted(
        list(drive_dir.glob('*.jpg')) +
        list(drive_dir.glob('*.JPG')) +
        list(drive_dir.glob('*.png'))
    )

stats_imgs = []

for scene in SCENES:
    imgs = get_scene_imgs(scene)
    if not imgs:
        print(f'[{scene}] sin imágenes en Drive — ejecuta primero a-dataset')
        continue
    n = len(imgs)

    with PILImage.open(imgs[0]) as im:
        w, h = im.size

    brightnesses, file_sizes = [], []
    for f in imgs[::5]:
        with PILImage.open(f) as im:
            arr = np.array(im.convert('L'), dtype=float)
            brightnesses.append(arr.mean())
        file_sizes.append(f.stat().st_size / 1024)

    stats_imgs.append({
        'Escena'          : scene,
        'N imágenes'      : n,
        'Resolución'      : f'{w}×{h}',
        'Megapíxeles'     : round(w * h / 1e6, 2),
        'Brillo medio'    : round(float(np.mean(brightnesses)), 1),
        'Std brillo'      : round(float(np.std(brightnesses)), 1),
        'Tamaño med (KB)' : round(float(np.mean(file_sizes)), 0),
    })

df_imgs = pd.DataFrame(stats_imgs)
print("Estadísticas básicas de imágenes (resolución downscale ×4):")
display(df_imgs)

Estadísticas básicas de imágenes (resolución downscale ×4):


,Escena,N imágenes,Resolución,Megapíxeles,Brillo medio,Std brillo,Tamaño med (KB)
0,garden,185,1297×840,1.09,111.3,7.4,1027.0
1,bicycle,194,1237×822,1.02,89.7,27.7,857.0
2,bonsai,292,780×520,0.41,69.8,20.9,277.0
3,counter,240,779×519,0.40,81.2,11.6,304.0


**Análisis de variación de condiciones de captura:**
El desvío estándar del brillo entre imágenes (`Std brillo`) es un indicador de la variación
de condiciones durante la captura. Valores altos sugieren cambios de exposición o iluminación
que pueden dificultar el matching de features entre frames.

# (c) Visualización de la trayectoria de cámara (poses oficiales)

El centro óptico de cada cámara se recupera como $\mathbf{c} = -\mathbf{R}^\top \mathbf{t}$,
donde $\mathbf{R}$ y $\mathbf{t}$ provienen de la pose `cam_from_world` almacenada en la
reconstrucción COLMAP oficial del dataset (`sparse/0/`).

In [ ]:
def get_cam_center(image):

    # Extrae el centro óptico de la cámara en coordenadas mundo.
    # Fórmula: c = -R^T t   donde (R, t) son la rotación y traslación de cam_from_world.

    if hasattr(image, 'cam_from_world'):
        cfw = image.cam_from_world
        if callable(cfw):
            cfw = cfw()
        rot = cfw.rotation
        R = np.array(rot.matrix() if callable(rot.matrix) else rot.matrix)
        t = np.array(cfw.translation)
    else:
        qw, qx, qy, qz = image.qvec
        R = np.array([
            [1-2*(qy**2+qz**2), 2*(qx*qy-qw*qz),  2*(qx*qz+qw*qy)],
            [2*(qx*qy+qw*qz),   1-2*(qx**2+qz**2), 2*(qy*qz-qw*qx)],
            [2*(qx*qz-qw*qy),   2*(qy*qz+qw*qx),   1-2*(qx**2+qy**2)],
        ])
        t = np.array(image.tvec)
    return -R.T @ t

SCENE_COLORS = {
    'garden': '#2ecc71', 'bicycle': '#3498db',
    'bonsai': '#e74c3c', 'counter': '#f39c12'
}

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=SCENES,
    specs=[[{'type': 'scatter3d'} for _ in range(2)] for _ in range(2)]
)

for idx, scene in enumerate(SCENES):
    official_dir = DRIVE_BASE / scene / 'sparse' / '0'

    if not official_dir.exists():
        print(f'[{scene}] reconstrucción oficial no encontrada en {official_dir}')
        continue

    rec = pycolmap.Reconstruction(str(official_dir))

    # Ordenar imágenes por nombre para trazar la trayectoria en secuencia
    ordered_ids = sorted(rec.images.keys(), key=lambda k: rec.images[k].name)
    centers = np.array([get_cam_center(rec.images[k]) for k in ordered_ids])

    row, col = idx // 2 + 1, idx % 2 + 1
    fig.add_trace(go.Scatter3d(
        x=centers[:, 0], y=centers[:, 1], z=centers[:, 2],
        mode='markers+lines',
        marker=dict(size=3, color=SCENE_COLORS[scene]),
        line=dict(color=SCENE_COLORS[scene], width=1),
        name=scene
    ), row=row, col=col)

    print(f'[{scene}] {len(centers)} cámaras graficadas')

fig.update_layout(
    height=800,
    title_text='Trayectorias de cámara — poses oficiales mip-NeRF 360',
    showlegend=True
)
fig.show()

[garden] 185 cámaras graficadas
[bicycle] 194 cámaras graficadas
[bonsai] 292 cámaras graficadas
[counter] 240 cámaras graficadas


Los gráficos se pueden ver correctamente en Google Colab. Por alguna razón no se logran ver en VScode

## (d) Estadísticas de la reconstrucción COLMAP oficial

In [ ]:
stats_official = []

for scene in SCENES:
    off_path = DRIVE_BASE / scene / 'sparse' / '0'
    n_total  = len(get_scene_imgs(scene))

    if not off_path.exists():
        print(f'[{scene}] reconstrucción oficial no encontrada')
        stats_official.append({'Escena': scene, 'Imgs. registradas': 'N/A',
                                '% reg.': 'N/A', 'Puntos 3D': 'N/A',
                                'Error reproyección (px)': 'N/A',
                                'Pts 3D / img reg.': 'N/A'})
        continue

    rec   = pycolmap.Reconstruction(str(off_path))
    n_reg = rec.num_reg_images()
    n_pts = rec.num_points3D()

    errors   = [p.error for p in rec.points3D.values() if p.error > 0]
    mean_err = float(np.mean(errors)) if errors else float('nan')

    stats_official.append({
        'Escena'                  : scene,
        'Imgs. registradas'       : n_reg,
        '% reg.'                  : round(100.0 * n_reg / n_total, 1) if n_total else 'N/A',
        'Puntos 3D'               : n_pts,
        'Error reproyección (px)' : round(mean_err, 3),
        'Pts 3D / img reg.'       : round(n_pts / n_reg, 0) if n_reg else 'N/A',
    })
    print(f'[{scene}] {n_reg}/{n_total} imgs | {n_pts:,} pts3D | err={mean_err:.3f}px')

df_off = pd.DataFrame(stats_official)
print("\nEstadísticas de la reconstrucción oficial:")
display(df_off)

[garden] 185/185 imgs | 138,766 pts3D | err=1.197px
[bicycle] 194/194 imgs | 54,275 pts3D | err=1.200px
[bonsai] 292/292 imgs | 206,613 pts3D | err=0.774px
[counter] 240/240 imgs | 155,767 pts3D | err=0.740px

Estadísticas de la reconstrucción oficial:


,Escena,Imgs. registradas,% reg.,Puntos 3D,Error reproyección (px),Pts 3D / img reg.
0,garden,185,100.0,138766,1.197,750.0
1,bicycle,194,100.0,54275,1.200,280.0
2,bonsai,292,100.0,206613,0.774,708.0
3,counter,240,100.0,155767,0.740,649.0


**Interpretación de métricas:**
- **% registradas**: fracción de imágenes que COLMAP logró ubicar en la reconstrucción.
- **Puntos 3D**: densidad de la nube sparse. Escenas con más textura producen más puntos.
- **Error de reproyección**: error medio (en píxeles) al proyectar los puntos 3D de vuelta
  a las imágenes. Valores típicos para COLMAP son 0.5–1.5 px, valores mayores indican
  problemas de calibración o features poco confiables.
- **Pts 3D / img reg.**: densidad relativa. Captura cuánta información 3D aporta cada imagen.